# q01：音声収録結果の確認と音声区間アノテーション

基礎レベル1では，サンプリング周波数 $F_s=16000$ Hzで収録した音声を読み込み，音声区間をアノテーションする．


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display



from pathlib import Path

ROOT = Path("./data")
INPUT_DIR = ROOT / "inputs"
ANNOTATION_DIR = ROOT / "annotations"
RESULT_DIR = ROOT / "results"

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_IDS = [f"A{i}" for i in range(1, 7)]
TEST_IDS = [f"A{i}" for i in range(7, 10)]
ALL_IDS = TRAIN_IDS + TEST_IDS

FRAME_SEC = 0.010
TARGET_FS = 16000

def wav_path(utt_id: str) -> Path:
    return INPUT_DIR / f"{utt_id}.wav"

def interval_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_intervals.csv"

def label_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_labels.csv"



import numpy as np
import pandas as pd
import soundfile as sf

EPS = 1e-12

def load_audio(path, target_fs: int = 16000):
    """WAVを読み込み，モノラルfloat32波形として返す．"""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"音声ファイルが見つかりません: {path}")
    x, fs = sf.read(path, dtype="float32")
    if x.ndim == 2:
        x = x.mean(axis=1)
    if fs != target_fs:
        raise ValueError(f"{path} のサンプリング周波数が {fs} Hz です．{target_fs} Hzで収録した音声を使用してください．")
    return x.astype(np.float32), fs

def frame_signal(x: np.ndarray, fs: int, frame_sec: float = 0.010):
    """10 msごとに非重複フレーム化する．不足分は0で埋める．"""
    frame_len = int(round(frame_sec * fs))
    n_frames = int(np.ceil(len(x) / frame_len))
    pad_len = n_frames * frame_len - len(x)
    if pad_len > 0:
        x = np.pad(x, (0, pad_len))
    frames = x.reshape(n_frames, frame_len)
    times = np.arange(n_frames) * frame_sec
    return frames, times

def hann_window(n: int):
    return np.hanning(n).astype(np.float32)

def short_time_log_energy(x: np.ndarray, fs: int, frame_sec: float = 0.010, use_hann: bool = True):
    frames, times = frame_signal(x, fs, frame_sec)
    if use_hann:
        frames = frames * hann_window(frames.shape[1])[None, :]
    energy = np.mean(frames ** 2, axis=1)
    log_energy = 10.0 * np.log10(EPS + energy)
    return log_energy, times

def relative_energy(log_energy: np.ndarray, percentile: float = 20.0):
    noise_floor = np.percentile(log_energy, percentile)
    score = log_energy - noise_floor
    return score, noise_floor

def apply_hangover(pred: np.ndarray, hangover_frames: int):
    """1から0へ切り替わった直後の指定フレーム数を1にする．"""
    pred = np.asarray(pred, dtype=np.int64).copy()
    if hangover_frames <= 0:
        return pred
    out = pred.copy()
    n = len(pred)
    for m in range(n - 1):
        if pred[m] == 1 and pred[m + 1] == 0:
            end = min(n, m + 1 + hangover_frames)
            out[m + 1:end] = 1
    return out

def labels_from_intervals(intervals, n_frames: int, frame_sec: float = 0.010):
    """[(start_sec, end_sec), ...]を10 msフレームラベルへ変換する．"""
    labels = np.zeros(n_frames, dtype=np.int64)
    for start_sec, end_sec in intervals:
        start = max(0, int(np.floor(start_sec / frame_sec)))
        end = min(n_frames, int(np.ceil(end_sec / frame_sec)))
        labels[start:end] = 1
    return labels

def save_labels_from_intervals(utt_id: str, intervals, duration_sec: float):
    """区間アノテーションCSVとフレームラベルCSVを保存する．"""
    n_frames = int(np.ceil(duration_sec / FRAME_SEC))
    interval_df = pd.DataFrame(intervals, columns=["start_sec", "end_sec"])
    interval_df["label"] = 1
    interval_df.to_csv(interval_path(utt_id), index=False)

    labels = labels_from_intervals(intervals, n_frames, FRAME_SEC)
    label_df = pd.DataFrame({
        "frame": np.arange(n_frames),
        "start_sec": np.arange(n_frames) * FRAME_SEC,
        "end_sec": (np.arange(n_frames) + 1) * FRAME_SEC,
        "label": labels
    })
    label_df.to_csv(label_path(utt_id), index=False)
    return interval_df, label_df

def load_frame_labels(utt_id: str, n_frames: int):
    """q01で保存したフレームラベルを読み込む．長さが合わない場合は切り詰めまたは0埋めする．"""
    path = label_path(utt_id)
    if not path.exists():
        raise FileNotFoundError(f"アノテーションが見つかりません: {path}")
    labels = pd.read_csv(path)["label"].to_numpy(dtype=np.int64)
    if len(labels) < n_frames:
        labels = np.pad(labels, (0, n_frames - len(labels)))
    elif len(labels) > n_frames:
        labels = labels[:n_frames]
    return labels

def confusion_binary(y_true: np.ndarray, y_pred: np.ndarray, positive_label: int = 1):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == positive_label) & (y_pred == positive_label)))
    fp = int(np.sum((y_true != positive_label) & (y_pred == positive_label)))
    fn = int(np.sum((y_true == positive_label) & (y_pred != positive_label)))
    tn = int(np.sum((y_true != positive_label) & (y_pred != positive_label)))
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def f1_from_counts(tp: int, fp: int, fn: int):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall

def macro_f1_score(y_true: np.ndarray, y_pred: np.ndarray):
    """音声クラスと非音声クラスのF1平均を返す．"""
    c_s = confusion_binary(y_true, y_pred, positive_label=1)
    f1_s, p_s, r_s = f1_from_counts(c_s["TP"], c_s["FP"], c_s["FN"])

    c_n = confusion_binary(1 - y_true, 1 - y_pred, positive_label=1)
    f1_n, p_n, r_n = f1_from_counts(c_n["TP"], c_n["FP"], c_n["FN"])

    return {
        "macro_f1": (f1_s + f1_n) / 2,
        "f1_speech": f1_s,
        "precision_speech": p_s,
        "recall_speech": r_s,
        "f1_non_speech": f1_n,
        "precision_non_speech": p_n,
        "recall_non_speech": r_n,
        **c_s
    }

def load_dataset(utt_ids):
    """音声，エネルギー，正解ラベルをIDごとに読み込む．"""
    data = {}
    for utt_id in utt_ids:
        x, fs = load_audio(wav_path(utt_id), TARGET_FS)
        E, times = short_time_log_energy(x, fs, FRAME_SEC, use_hann=True)
        y = load_frame_labels(utt_id, len(E))
        data[utt_id] = {"x": x, "fs": fs, "E": E, "times": times, "y": y}
    return data

def concat_labels_and_preds(y_list, pred_list):
    return np.concatenate(y_list), np.concatenate(pred_list)

## 1．A1.wavの読み込み

A1の音声を読み込み，長さとサンプリング周波数を確認する．

In [ ]:
utt_id = "A1"
x, fs = load_audio(wav_path(utt_id), TARGET_FS)
duration_sec = len(x) / fs

print(f"file: {wav_path(utt_id)}")
print(f"sampling rate: {fs} Hz")
print(f"samples: {len(x)}")
print(f"duration: {duration_sec:.3f} sec")

display(Audio(x, rate=fs))

## 2．波形を表示

音声区間を目視で決めるため，波形を秒単位で表示する．

In [ ]:
t = np.arange(len(x)) / fs

plt.figure(figsize=(14, 4))
plt.plot(t, x)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.title(f"Waveform: {utt_id}")
plt.grid(True)
plt.show()

## 3．音声区間の入力

`speech_intervals` に音声が存在する区間を秒単位で入力する．  
例：0.35秒から1.80秒，2.10秒から4.50秒が音声なら `[(0.35, 1.80), (2.10, 4.50)]` とする．

In [ ]:
# ここを自分のA1.wavに合わせて編集する
speech_intervals = [
    # (start_sec, end_sec),
    # 例: (0.35, 1.80),
]

interval_df, label_df = save_labels_from_intervals(utt_id, speech_intervals, duration_sec)

print("区間アノテーション保存先:", interval_path(utt_id))
display(interval_df)

print("フレームラベル保存先:", label_path(utt_id))
display(label_df.head())
display(label_df.tail())

## 4．アノテーション結果の確認

波形の上に音声区間を重ねて表示する．

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(t, x, label="waveform")

for start_sec, end_sec in speech_intervals:
    plt.axvspan(start_sec, end_sec, alpha=0.25)

plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.title(f"Annotation check: {utt_id}")
plt.grid(True)
plt.show()

n_speech = int(label_df["label"].sum())
n_total = len(label_df)
print(f"speech frames: {n_speech}/{n_total}")
print(f"speech ratio: {n_speech / n_total:.3f}")

## 5．A1からA9までをまとめてアノテーションする場合

A2からA9までのWAVを同じフォルダに置いた場合，以下の辞書を編集して一括保存できる．  
q02以降では，A1からA6をtrain，A7からA9をtestとして利用する．

In [ ]:
# 必要に応じて編集する．未入力のIDは保存しない．
all_speech_intervals = {
    "A1": speech_intervals,
    # "A2": [(start_sec, end_sec)],
    # "A3": [(start_sec, end_sec)],
    # "A4": [(start_sec, end_sec)],
    # "A5": [(start_sec, end_sec)],
    # "A6": [(start_sec, end_sec)],
    # "A7": [(start_sec, end_sec)],
    # "A8": [(start_sec, end_sec)],
    # "A9": [(start_sec, end_sec)],
}

for uid, intervals in all_speech_intervals.items():
    if not wav_path(uid).exists():
        print(f"skip: {wav_path(uid)} が存在しません")
        continue
    x_i, fs_i = load_audio(wav_path(uid), TARGET_FS)
    duration_i = len(x_i) / fs_i
    save_labels_from_intervals(uid, intervals, duration_i)
    print(f"saved: {uid}")